In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import pypsa
from pathlib import Path
import networkx as nx
import numpy as np
import logging

logging.basicConfig(level=logging.WARNING)

In [2]:
import polars as pl
print(pl.__version__)

1.40.0


In [3]:
YEAR = 2019

BASE_PATH     = "data"
DENMARK_PATH  = f"{BASE_PATH}/Denmark_En_DH"
GERMANY_PATH  = f"{BASE_PATH}/Germany_En_DH"
NORWAY_PATH   = f"{BASE_PATH}/Norway_En_DH"
SWEDEN_PATH   = f"{BASE_PATH}/Sweden_En_DH"
LOAD_PATH     = f"{BASE_PATH}/Load/time_series_60min_singleindex.csv"

In [4]:
def annualize(capex, r, lifetime):
    crf = (r * (1 + r)**lifetime) / ((1 + r)**lifetime - 1)
    return capex * crf

r = 0.07

# Solar
CAPITAL_COST_SOLAR_DE = float(annualize(700_000,   r, lifetime=25))
CAPITAL_COST_SOLAR_DK = float(annualize(772_000,   r, lifetime=25))
CAPITAL_COST_SOLAR_SE = float(annualize(861_000,   r, lifetime=25))

# Wind onshore
CAPITAL_COST_WIND_ON_DE = float(annualize(1_900_000, r, lifetime=25))
CAPITAL_COST_WIND_ON_DK = float(annualize(1_950_000, r, lifetime=25))
CAPITAL_COST_WIND_ON_SE = float(annualize(1_400_000, r, lifetime=25))
CAPITAL_COST_WIND_ON_NO = float(annualize(1_400_000, r, lifetime=25))

# Wind offshore
CAPITAL_COST_WIND_OFF_DE = float(annualize(4_150_000, r, lifetime=25))
CAPITAL_COST_WIND_OFF_DK = float(annualize(2_900_000, r, lifetime=25))

# Gas DK
CAPITAL_COST_GAS_DK = float(annualize(1_300_000, r, lifetime=25))

# Marginal costs DK renewables
MARGINAL_COST_SOLAR_DK   = 0
MARGINAL_COST_WIND_ON_DK = 0
MARGINAL_COST_WIND_OFF_DK = 0
MARGINAL_COST_GAS_DK     = 43.18 / 0.47

In [5]:
snapshots = pd.date_range(
    start=f"{YEAR}-01-01 00:00:00",
    end=f"{YEAR}-12-31 23:00:00",
    freq="h"
)

pv       = pd.read_csv(f"{DENMARK_PATH}/ninja-pv-country-DK-national-merra2.csv", skiprows=3)
wind_on  = pd.read_csv(f"{DENMARK_PATH}/ninja-wind-country-DK-current_onshore-merra2.csv", skiprows=3)
wind_off = pd.read_csv(f"{DENMARK_PATH}/ninja-wind-country-DK-current_offshore-merra2.csv", skiprows=3)
load_df  = pd.read_csv(LOAD_PATH)

pv["time"]       = pd.to_datetime(pv["time"], utc=True)
wind_on["time"]  = pd.to_datetime(wind_on["time"], utc=True)
wind_off["time"] = pd.to_datetime(wind_off["time"], utc=True)
load_df["utc_timestamp"] = pd.to_datetime(load_df["utc_timestamp"], utc=True)

pv       = pv.set_index("time")
wind_on  = wind_on.set_index("time")
wind_off = wind_off.set_index("time")
load_df  = load_df.set_index("utc_timestamp")

pv.index       = pv.index.tz_convert(None)
wind_on.index  = wind_on.index.tz_convert(None)
wind_off.index = wind_off.index.tz_convert(None)
load_df.index  = load_df.index.tz_convert(None)

pv       = pv.loc[f"{YEAR}-01-01":f"{YEAR}-12-31 23:00:00"]
wind_on  = wind_on.loc[f"{YEAR}-01-01":f"{YEAR}-12-31 23:00:00"]
wind_off = wind_off.loc[f"{YEAR}-01-01":f"{YEAR}-12-31 23:00:00"]
load_df  = load_df.loc[f"{YEAR}-01-01":f"{YEAR}-12-31 23:00:00"]

data = pd.DataFrame(index=snapshots)
data["load"]        = load_df["DK_load_actual_entsoe_transparency"]
data["solar_cf"]    = pv["NATIONAL"]
data["wind_on_cf"]  = wind_on["NATIONAL"]
data["wind_off_cf"] = wind_off["NATIONAL"]

data = data.reindex(snapshots).interpolate(method="time").ffill().bfill()

print("DK NaN check:", data.isna().sum().sum())
print(data.head())

DK NaN check: 0
                        load  solar_cf  wind_on_cf  wind_off_cf
2019-01-01 00:00:00  3186.04       0.0    0.805840     0.896970
2019-01-01 01:00:00  3070.07       0.0    0.839171     0.916942
2019-01-01 02:00:00  2966.19       0.0    0.861352     0.927392
2019-01-01 03:00:00  2933.48       0.0    0.882439     0.948853
2019-01-01 04:00:00  2940.64       0.0    0.903024     0.960074


In [6]:
def read_load_energycharts(path, year):
    df = pd.read_csv(path, skiprows=[1])
    df.columns = df.columns.str.strip()
    df = df.rename(columns={df.columns[0]: "time"})
    df["time"] = pd.to_datetime(df["time"], utc=True).dt.tz_convert(None)
    df = df.set_index("time")
    df = df.loc[f"{year}-01-01":f"{year}-12-31 23:00:00"]
    load = df["Load"].astype(float)
    if len(load) > 9000:
        load = load.resample("h").mean()
    load = load[~load.index.duplicated(keep="first")]
    load = load.resample("h").mean().interpolate(method="time")
    return load


def read_country_data(pv_path, wind_on_path, wind_off_path, load_path, year):
    pv_c       = pd.read_csv(pv_path, skiprows=3)
    wind_on_c  = pd.read_csv(wind_on_path, skiprows=3)
    wind_off_c = pd.read_csv(wind_off_path, skiprows=3)

    for df in [pv_c, wind_on_c, wind_off_c]:
        df["time"] = pd.to_datetime(df["time"], utc=True)

    pv_c       = pv_c.set_index("time")
    wind_on_c  = wind_on_c.set_index("time")
    wind_off_c = wind_off_c.set_index("time")

    pv_c.index       = pv_c.index.tz_convert(None)
    wind_on_c.index  = wind_on_c.index.tz_convert(None)
    wind_off_c.index = wind_off_c.index.tz_convert(None)

    pv_c       = pv_c.loc[f"{year}-01-01":f"{year}-12-31 23:00:00"]
    wind_on_c  = wind_on_c.loc[f"{year}-01-01":f"{year}-12-31 23:00:00"]
    wind_off_c = wind_off_c.loc[f"{year}-01-01":f"{year}-12-31 23:00:00"]

    d = pd.DataFrame(index=snapshots)
    d["load"]        = read_load_energycharts(load_path, year).reindex(snapshots)
    d["solar_cf"]    = pv_c["NATIONAL"].astype(float)
    d["wind_on_cf"]  = wind_on_c["NATIONAL"].astype(float)
    d["wind_off_cf"] = wind_off_c["NATIONAL"].astype(float)

    return d.reindex(snapshots).interpolate(method="time").ffill().bfill()


data_de = read_country_data(
    pv_path       = f"{GERMANY_PATH}/ninja-pv-country-DE-national-merra2.csv",
    wind_on_path  = f"{GERMANY_PATH}/ninja-wind-country-DE-current_onshore-merra2.csv",
    wind_off_path = f"{GERMANY_PATH}/ninja-wind-country-DE-current_offshore-merra2.csv",
    load_path     = f"{GERMANY_PATH}/energy-charts_Public_net_electricity_generation_in_Germany_in_2019.csv",
    year=YEAR
)

data_no = read_country_data(
    pv_path       = f"{NORWAY_PATH}/ninja-pv-country-NO-national-merra2.csv",
    wind_on_path  = f"{NORWAY_PATH}/ninja-wind-country-NO-current_onshore-merra2.csv",
    wind_off_path = f"{NORWAY_PATH}/ninja-wind-country-NO-current_offshore-merra2.csv",
    load_path     = f"{NORWAY_PATH}/energy-charts_Public_net_electricity_generation_in_Norway_in_2019.csv",
    year=YEAR
)

data_se = read_country_data(
    pv_path       = f"{SWEDEN_PATH}/ninja-pv-country-SE-national-merra2.csv",
    wind_on_path  = f"{SWEDEN_PATH}/ninja-wind-country-SE-current_onshore-merra2.csv",
    wind_off_path = f"{SWEDEN_PATH}/ninja-wind-country-SE-current_offshore-merra2.csv",
    load_path     = f"{SWEDEN_PATH}/energy-charts_Public_net_electricity_generation_in_Sweden_in_2019.csv",
    year=YEAR
)

print("DE:", data_de.shape, data_de.isna().sum().sum())
print("NO:", data_no.shape, data_no.isna().sum().sum())
print("SE:", data_se.shape, data_se.isna().sum().sum())

DE: (8760, 4) 0
NO: (8760, 4) 0
SE: (8760, 4) 0


In [7]:
def read_hydro(path, hydro_cols, p_nom, year, resample_hourly=False):
    col_names = pd.read_csv(path, nrows=0).columns.tolist()
    df = pd.read_csv(path, skiprows=2, header=None, names=col_names,
                     parse_dates=[0], index_col=0)
    df.index = pd.to_datetime(df.index, utc=True).tz_convert(None)
    df = df.loc[f"{year}-01-01":f"{year}-12-31 23:00:00"]

    hydro_total = df[hydro_cols].sum(axis=1)
    if resample_hourly:
        hydro_total = hydro_total.resample("h").mean()

    d = pd.DataFrame(index=snapshots)
    d["hydro_cf"] = (hydro_total / p_nom).clip(0, 1)
    return d.reindex(snapshots).interpolate(method="time").ffill().bfill()


hydro_de = read_hydro(
    f"{GERMANY_PATH}/energy-charts_Public_net_electricity_generation_in_Germany_in_2019.csv",
    hydro_cols=["Hydro Run-of-River", "Hydro water reservoir", "Hydro pumped storage"],
    p_nom=5900, year=YEAR, resample_hourly=True
)

hydro_no_ror = read_hydro(
    f"{NORWAY_PATH}/energy-charts_Public_net_electricity_generation_in_Norway_in_2019.csv",
    hydro_cols=["Hydro Run of River"],
    p_nom=6300, year=YEAR
)

hydro_no_res = read_hydro(
    f"{NORWAY_PATH}/energy-charts_Public_net_electricity_generation_in_Norway_in_2019.csv",
    hydro_cols=["Hydro water reservoir"],
    p_nom=27530, year=YEAR
)

hydro_se = read_hydro(
    f"{SWEDEN_PATH}/energy-charts_Public_net_electricity_generation_in_Sweden_in_2019.csv",
    hydro_cols=["Hydro water reservoir"],
    p_nom=16320, year=YEAR
)

print("Hydro DE:", hydro_de.shape)
print("Hydro NO RoR:", hydro_no_ror.shape)
print("Hydro NO Res:", hydro_no_res.shape)
print("Hydro SE:", hydro_se.shape)

Hydro DE: (8760, 1)
Hydro NO RoR: (8760, 1)
Hydro NO Res: (8760, 1)
Hydro SE: (8760, 1)


In [8]:
LINE_DK_DE = 2100
LINE_DK_NO = 1700
LINE_DK_SE = 2400
LINE_NO_SE = 3700
LINE_DE_SE = 600

nd = pypsa.Network()
nd.set_snapshots(snapshots)

# --- Buses ---
nd.add("Bus", "Denmark", v_nom=400)
nd.add("Bus", "Germany", v_nom=400)
nd.add("Bus", "Norway",  v_nom=400)
nd.add("Bus", "Sweden",  v_nom=400)

# --- Lines ---
nd.add("Line", "DK-DE", bus0="Denmark", bus1="Germany", x=0.1, s_nom=LINE_DK_DE, s_nom_extendable=False)
nd.add("Line", "DK-NO", bus0="Denmark", bus1="Norway",  x=0.1, s_nom=LINE_DK_NO, s_nom_extendable=False)
nd.add("Line", "DK-SE", bus0="Denmark", bus1="Sweden",  x=0.1, s_nom=LINE_DK_SE, s_nom_extendable=False)
nd.add("Line", "NO-SE", bus0="Norway",  bus1="Sweden",  x=0.1, s_nom=LINE_NO_SE, s_nom_extendable=False)
nd.add("Line", "DE-SE", bus0="Germany", bus1="Sweden",  x=0.1, s_nom=LINE_DE_SE, s_nom_extendable=False)

# --- Denmark (extendable) ---
nd.add("Load", "DK_demand", bus="Denmark", p_set=data["load"].values)
nd.add("Generator", "DK_solar",    bus="Denmark", carrier="solar",
       p_max_pu=data["solar_cf"].values, p_nom_extendable=True,
       capital_cost=CAPITAL_COST_SOLAR_DK, marginal_cost=MARGINAL_COST_SOLAR_DK)
nd.add("Generator", "DK_wind_on",  bus="Denmark", carrier="wind_onshore",
       p_max_pu=data["wind_on_cf"].values, p_nom_extendable=True,
       capital_cost=CAPITAL_COST_WIND_ON_DK, marginal_cost=MARGINAL_COST_WIND_ON_DK)
nd.add("Generator", "DK_wind_off", bus="Denmark", carrier="wind_offshore",
       p_max_pu=data["wind_off_cf"].values, p_nom_extendable=True,
       capital_cost=CAPITAL_COST_WIND_OFF_DK, marginal_cost=MARGINAL_COST_WIND_OFF_DK)

# --- Germany ---
nd.add("Load", "DE_demand", bus="Germany", p_set=data_de["load"].values)
nd.add("Generator", "DE_wind_onshore", bus="Germany", carrier="wind_onshore",
       p_max_pu=data_de["wind_on_cf"].values,
       p_nom=53100, p_nom_min=53100, p_nom_extendable=True,
       marginal_cost=2, capital_cost=CAPITAL_COST_WIND_ON_DE)
nd.add("Generator", "DE_wind_off",     bus="Germany", carrier="wind_offshore",
       p_max_pu=data_de["wind_off_cf"].values,
       p_nom=7700, p_nom_min=7700, p_nom_extendable=True,
       marginal_cost=5, capital_cost=CAPITAL_COST_WIND_OFF_DE)
nd.add("Generator", "DE_solar",        bus="Germany", carrier="solar",
       p_max_pu=data_de["solar_cf"].values,
       p_nom=46200, p_nom_min=46200, p_nom_extendable=True,
       marginal_cost=1, capital_cost=CAPITAL_COST_SOLAR_DE)
nd.add("Generator", "DE_lignite",      bus="Germany", carrier="lignite",
       p_nom=25900, p_nom_extendable=False, marginal_cost=40.6)
nd.add("Generator", "DE_hard_coal",    bus="Germany", carrier="coal",
       p_nom=22670, p_nom_extendable=False, marginal_cost=38.8)
nd.add("Generator", "DE_biomass",      bus="Germany", carrier="biomass",
       p_nom=8270,  p_nom_extendable=False, marginal_cost=57.2)
nd.add("Generator", "DE_hydro",        bus="Germany", carrier="hydro",
       p_nom=4800,  p_nom_extendable=False, marginal_cost=5,
       p_max_pu=hydro_de["hydro_cf"].values)
nd.add("Generator", "DE_nuclear",      bus="Germany", carrier="nuclear",
       p_nom=9500,  p_nom_extendable=False, marginal_cost=8)

# --- Norway ---
nd.add("Load", "NO_demand", bus="Norway", p_set=data_no["load"].values)
nd.add("Generator", "NO_hydro_ror", bus="Norway", carrier="hydro",
       p_nom=6300,  p_nom_extendable=False,
       marginal_cost=3, p_max_pu=hydro_no_ror["hydro_cf"].values)
nd.add("Generator", "NO_hydro_res", bus="Norway", carrier="hydro",
       p_nom=27530, p_nom_extendable=False,
       marginal_cost=2, p_max_pu=hydro_no_res["hydro_cf"].values)
nd.add("Generator", "NO_wind",      bus="Norway", carrier="wind_onshore",
       p_max_pu=data_no["wind_on_cf"].values,
       p_nom=3900, p_nom_min=3900, p_nom_extendable=True,
       marginal_cost=2, capital_cost=CAPITAL_COST_WIND_ON_NO)

# --- Sweden ---
nd.add("Load", "SE_demand", bus="Sweden", p_set=data_se["load"].values)
nd.add("Generator", "SE_hydro",   bus="Sweden", carrier="hydro",
       p_nom=16320, p_nom_extendable=False,
       marginal_cost=3, p_max_pu=hydro_se["hydro_cf"].values)
nd.add("Generator", "SE_nuclear", bus="Sweden", carrier="nuclear",
       p_nom=7710, p_nom_extendable=False, marginal_cost=8)
nd.add("Generator", "SE_wind",    bus="Sweden", carrier="wind_onshore",
       p_max_pu=data_se["wind_on_cf"].values,
       p_nom=9650, p_nom_min=9650, p_nom_extendable=True,
       marginal_cost=2, capital_cost=CAPITAL_COST_WIND_ON_SE)
nd.add("Generator", "SE_solar",   bus="Sweden", carrier="solar",
       p_max_pu=data_se["solar_cf"].values,
       p_nom=1100, p_nom_min=1100, p_nom_extendable=True,
       marginal_cost=1, capital_cost=CAPITAL_COST_SOLAR_SE)

In [9]:
G = nx.Graph()
for line, row in nd.lines.iterrows():
    G.add_edge(row.bus0, row.bus1, name=line)

cycles = nx.cycle_basis(G)
print(f"Nº cycles: {len(cycles)}")
for i, cycle in enumerate(cycles):
    print(f"  Cycle {i+1}: {' → '.join(cycle)} → {cycle[0]}")

print(f"Connected network: {nx.is_connected(G)}")

Nº cycles: 2
  Cycle 1: Denmark → Germany → Sweden → Denmark
  Cycle 2: Denmark → Norway → Sweden → Denmark
Connected network: True


**PART G**

In [10]:

CAPITAL_COST_ELECTROLYZER = float(annualize(903_906, r, lifetime=40))   # €/MW
CAPITAL_COST_H2_STORAGE   = float(annualize(6_010,  r, lifetime=20))    # €/MWh
CAPITAL_COST_FUEL_CELL = float(annualize(1_591_578, r, lifetime=30))    # €/MW

CAPITAL_COST_CH4_METHA = float(annualize(979_540, r, lifetime=30))   # €/MW
CAPITAL_COST_CH4_STORAGE   = float(annualize(2_974,  r, lifetime=20))    # €/MWh
CAPITAL_COST_CH4_TURBINE = float(annualize(606_401, r, lifetime=25))

CH4_DISTANCES_KM = H2_DISTANCES_KM = {
    ("Denmark", "Germany"): 350,
    ("Denmark", "Norway"):  500,
    ("Denmark", "Sweden"):  300,
    ("Germany", "Sweden"):  800,
    ("Germany", "Norway"):  700,
}


In [11]:
# --- 0. CONFIGURATION ---
countries = {"DK": "Denmark", "DE": "Germany", "NO": "Norway", "SE": "Sweden"}


# ─── 1. BUSES (H2 and CH4) ──────────────────────────────────────────
for code, name in countries.items():
    nd.add("Bus", f"{name} H2", carrier="H2")
    nd.add("Bus", f"{name} CH4", carrier="CH4")


# ─── 2. H2 SYSTEM ───────────────────────────────────────────────────
for code, name in countries.items():
    # Electrolyzers
    nd.add(
        "Link", f"{code} electrolyzer",
        bus0=name, bus1=f"{name} H2",
        carrier="H2",
        p_nom_extendable=True,
        efficiency=0.58,
        capital_cost=CAPITAL_COST_ELECTROLYZER
    )

    # H2 Storage
    nd.add(
        "Store", f"{code} H2 store",
        bus=f"{name} H2",
        carrier="H2",
        e_nom_extendable=True,
        standing_loss=0.001,
        capital_cost=CAPITAL_COST_H2_STORAGE
    )

    # Fuel Cells
    nd.add(
        "Link", f"{code} fuel cell",
        bus0=f"{name} H2", bus1=name,
        carrier="H2",
        p_nom_extendable=True,
        efficiency=0.4869,
        capital_cost=CAPITAL_COST_FUEL_CELL
    )


# H2 Pipelines
for (a, b), dist in H2_DISTANCES_KM.items():
    nd.add(
        "Link", f"H2 pipeline {a}-{b}",
        bus0=f"{a} H2", bus1=f"{b} H2",
        carrier="H2 pipeline",
        p_nom_extendable=True,
        capital_cost=0
    )
    nd.add(
        "Link", f"H2 pipeline {b}-{a}",
        bus0=f"{b} H2", bus1=f"{a} H2",
        carrier="H2 pipeline",
        p_nom_extendable=True,
        capital_cost=0
    )


# ─── 3. CH4 SYSTEM ──────────────────────────────────────────────────
market_costs = {"DK": 43.18, "DE": 34.78, "NO": 17.26, "SE": 45.35}


gas_market_cap = {
    "DK": None,
    "DE": 30700,
    "NO": 600,
    "SE": 8800
}

for code, name in countries.items():
    cap = gas_market_cap[code]
    is_extendable = cap is None

    # Gas Market
    nd.add(
        "Generator",
        f"{code} CH4 market",
        bus=f"{name} CH4",
        carrier="gas market",
        p_nom=0 if is_extendable else cap,
        p_nom_extendable=is_extendable,
        marginal_cost=market_costs.get(code, 40)
    )

    # Methanation
    nd.add(
        "Link",
        f"{code} methanation",
        bus0=name,
        bus1=f"{name} CH4",
        carrier="CH4",
        p_nom_extendable=True,
        efficiency=0.71,
        capital_cost=CAPITAL_COST_CH4_METHA
    )

    # CH4 Storage
    nd.add(
        "Store",
        f"{code} CH4 store",
        bus=f"{name} CH4",
        carrier="CH4",
        e_nom_extendable=True,
        standing_loss=0.001,
        capital_cost=CAPITAL_COST_CH4_STORAGE
    )


# Gas Plants (Links)
existing_gas_plants = {"DE": 30700, "NO": 600, "SE": 8800, "DK": None}

for code, pnom in existing_gas_plants.items():
    name = countries[code]
    is_extendable = pnom is None

    nd.add(
        "Link", f"{code}_gas",
        bus0=f"{name} CH4", bus1=name,
        carrier="gas",
        p_nom=0 if is_extendable else pnom,
        p_nom_extendable=is_extendable,
        efficiency=0.47,
        capital_cost=CAPITAL_COST_CH4_TURBINE if is_extendable else 0,
        marginal_cost=0
    )


# CH4 Pipelines
for (a, b), dist in CH4_DISTANCES_KM.items():
    eff = 0.995 ** (dist / 1000)

    nd.add(
        "Link", f"CH4 pipeline {a}-{b}",
        bus0=f"{a} CH4", bus1=f"{b} CH4",
        carrier="CH4 pipeline",
        p_nom_extendable=True,
        efficiency=eff,
        capital_cost=0
    )
    nd.add(
        "Link", f"CH4 pipeline {b}-{a}",
        bus0=f"{b} CH4", bus1=f"{a} CH4",
        carrier="CH4 pipeline",
        p_nom_extendable=True,
        efficiency=eff,
        capital_cost=0
    )

print(f"CH4/H2 system ready. Buses: {len(nd.buses)} | Links: {len(nd.links)} | Stores: {len(nd.stores)}")

CH4/H2 system ready. Buses: 12 | Links: 36 | Stores: 8


In [12]:

nd.optimize(solver_name="gurobi")  # o el que uses
print("Optimización H2 y CH4 lista:")

C:\Users\raula\AppData\Local\Temp\ipykernel_10168\612789415.py:1: FutureWarning: The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.
  nd.optimize(solver_name="gurobi")  # o el que uses
Index(['Denmark', 'Germany', 'Norway', 'Sweden', 'Denmark H2', 'Denmark CH4',
       'Germany H2', 'Germany CH4', 'Norway H2', 'Norway CH4', 'Sweden H2',
       'Sweden CH4'],
      dtype='str', name='name')
Index(['DK_solar', 'DK_wind_on', 'DK_wind_off', 'DE_wind_onshore',
       'DE_wind_off', 'DE_solar', 'DE_lignite', 'DE_hard_coal', 'DE_biomass',
       'DE_hydro', 'DE_nuclear', 'NO_hydro_ror', 'NO_hydro_res', 'NO_wind',
       'SE_hydro', 'SE_nuclear', 'SE_wind', 'SE_solar', 'DK CH4 market',
       'DE CH4 market', 'NO CH4 market', 'SE CH4 market'],
      dtype='str', name='name')
Inde

Set parameter Username


Set parameter LicenseID to value 2785388
Academic license - for non-commercial use only - expires 2027-02-27
Read LP format model from file C:\Users\raula\AppData\Local\Temp\linopy-problem-xcwnivrj.lp
Reading time = 2.77 seconds
obj: 1436691 rows, 692092 columns, 2921808 nonzeros
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 7 260 w/ Radeon 780M Graphics, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 1436691 rows, 692092 columns and 2921808 nonzeros (Min)
Model fingerprint: 0xbeb80a03
Model has 166471 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e-06, 1e+00]
  Objective range  [1e+00, 4e+05]
  Bounds range     [2e+10, 2e+10]
  RHS range        [6e+02, 8e+04]
         Consider reformulating model or setting NumericFocus parameter
         to avoid numerical issues.

Presolve removed 1090366 rows and 179298 

In [13]:
import pandas as pd


country_map = {
    "Denmark": "DK",
    "Germany": "DE",
    "Norway": "NO",
    "Sweden": "SE",
    "Denmark H2": "DK",
    "Germany H2": "DE",
    "Norway H2": "NO",
    "Sweden H2": "SE",
    "Denmark CH4": "DK",
    "Germany CH4": "DE",
    "Norway CH4": "NO",
    "Sweden CH4": "SE",
}


# =========================
# 1) GENERATORS
# =========================
gen = nd.generators.copy()
gen["country"] = gen["bus"].map(country_map)
gen = gen[gen["carrier"] != "gas market"]

gen_caps = gen.groupby(["country", "carrier"])["p_nom_opt"].sum().round(1)

gen_energy = (
    pd.concat(
        [
            gen[["country", "carrier"]],
            (nd.generators_t.p[gen.index].sum() / 1e6).rename("TWh")
        ],
        axis=1
    )
    .groupby(["country", "carrier"])["TWh"]
    .sum()
    .round(3)
)


# =========================
# 2) LINKS
# =========================
links = nd.links.copy()

def get_country(row):
    if row["bus0"] in country_map:
        return country_map[row["bus0"]]
    if row["bus1"] in country_map:
        return country_map[row["bus1"]]
    return None

links["country"] = links.apply(get_country, axis=1)

national_links = links[~links["carrier"].isin(["H2 pipeline", "CH4 pipeline"])]
national_link_caps = national_links.groupby(["country", "carrier"])["p_nom_opt"].sum().round(1)

national_link_energy = (
    pd.concat(
        [
            national_links[["country", "carrier"]],
            (nd.links_t.p0[national_links.index].abs().sum() / 1e6).rename("TWh")
        ],
        axis=1
    )
    .groupby(["country", "carrier"])["TWh"]
    .sum()
    .round(3)
)


# =========================
# 3) 
# =========================
all_country_caps = (
    pd.concat([gen_caps, national_link_caps])
    .groupby(level=[0,1])
    .sum()
    .sort_index()
)

all_country_energy = (
    pd.concat([gen_energy, national_link_energy])
    .groupby(level=[0,1])
    .sum()
    .sort_index()
)

all_country_summary = pd.concat(
    [all_country_caps.rename("MW"), all_country_energy.rename("TWh")],
    axis=1
).fillna(0).round({"MW": 1, "TWh": 3})


print("\n--- COUNTRY TECHNOLOGIES (MW / TWh-year) ---")
print(all_country_summary)


# =========================
# 4) Pipelines
# =========================
pipelines = links[links["carrier"].isin(["H2 pipeline", "CH4 pipeline"])]

pipeline_caps = pipelines.groupby("carrier")["p_nom_opt"].sum().round(1)

pipeline_energy = (
    pd.concat(
        [
            pipelines[["carrier"]],
            (nd.links_t.p0[pipelines.index].abs().sum() / 1e6).rename("TWh")
        ],
        axis=1
    )
    .groupby("carrier")["TWh"]
    .sum()
    .round(3)
)

pipeline_summary = pd.concat(
    [pipeline_caps.rename("MW"), pipeline_energy.rename("TWh")],
    axis=1
).fillna(0).round({"MW": 1, "TWh": 3})


print("\n--- PIPELINES (MW / TWh-year) ---")
print(pipeline_summary)


# =========================

# =========================
print("\n--- PIPELINES BY LINK (MW / TWh-year) ---")
for carrier in ["H2 pipeline", "CH4 pipeline"]:
    subset = pipelines[pipelines["carrier"] == carrier]
    if not subset.empty:
        sub = pd.DataFrame({
            "MW": subset["p_nom_opt"].round(1),
            "TWh": (nd.links_t.p0[subset.index].abs().sum() / 1e6).round(3)
        }).sort_values("MW", ascending=False)

        print(f"\n{carrier}:")
        print(sub)


--- COUNTRY TECHNOLOGIES (MW / TWh-year) ---
                            MW      TWh
country carrier                        
DE      CH4                0.0    0.000
        H2                 0.0    0.000
        biomass         8270.0    0.597
        coal           22670.0  170.226
        gas            30700.0    2.577
        hydro           4800.0   22.882
        lignite        25900.0   57.056
        nuclear         9500.0   82.015
        solar          46200.0   51.254
        wind_offshore   7700.0   22.473
        wind_onshore   53100.0   99.866
DK      CH4                0.0    0.000
        H2                 0.0    0.000
        gas             4235.9    3.185
        solar              0.0    0.000
        wind_offshore      0.0    0.000
        wind_onshore       0.0    0.000
NO      CH4                0.0    0.000
        H2               737.2    0.122
        gas              600.0    0.971
        hydro          33830.0  122.363
        wind_onshore    5642.7   1

In [14]:
import pandas as pd

# 1. generators
gen = nd.generators.copy()
gen["country"] = gen["bus"].map(country_map)
gen = gen[gen["carrier"] != "gas market"]

gen_energy = (nd.generators_t.p[gen.index].sum() / 1e6).rename("TWh")
gen_df = pd.concat([gen[["country", "carrier", "p_nom_opt"]], gen_energy], axis=1)

# 2. LINKS 
links = nd.links.copy()
links["country"] = links.apply(lambda row: country_map.get(row["bus0"], country_map.get(row["bus1"])), axis=1)

is_pipeline = links["carrier"].isin(["H2 pipeline", "CH4 pipeline"])
national_links = links[~is_pipeline].copy()

link_energy = (nd.links_t.p0[national_links.index].abs().sum() / 1e6).rename("TWh")
link_df = pd.concat([national_links[["country", "carrier", "p_nom_opt"]], link_energy], axis=1)

national_df = pd.concat([gen_df, link_df])
national_summary = national_df.groupby(["country", "carrier"]).agg({
    "p_nom_opt": "sum",
    "TWh": "sum"
}).round(2)

print("\n--- COUNTRY TECHNOLOGIES (MW / TWh-year) ---")
print(national_summary)

# 4. PIPELINES
pipelines = links[is_pipeline].copy()
pipeline_caps = pipelines.groupby("carrier")["p_nom_opt"].sum().round(1)
pipeline_energy = (nd.links_t.p0[pipelines.index].abs().sum() / 1e6).groupby(pipelines["carrier"]).sum().round(3)

pipeline_summary = pd.concat([pipeline_caps.rename("MW"), pipeline_energy.rename("TWh")], axis=1)

print("\n--- PIPELINES (MW / TWh-year) ---")
print(pipeline_summary)


--- COUNTRY TECHNOLOGIES (MW / TWh-year) ---
                       p_nom_opt     TWh
country carrier                         
DE      CH4                 0.00    0.00
        H2                  0.00    0.00
        biomass          8270.00    0.60
        coal            22670.00  170.23
        gas             30700.00    2.58
        hydro            4800.00   22.88
        lignite         25900.00   57.06
        nuclear          9500.00   82.02
        solar           46200.00   51.25
        wind_offshore    7700.00   22.47
        wind_onshore    53100.00   99.87
DK      CH4                 0.00    0.00
        H2                  0.00    0.00
        gas              4235.88    3.19
        solar               0.00    0.00
        wind_offshore       0.00    0.00
        wind_onshore        0.00    0.00
NO      CH4                 0.00    0.00
        H2                737.21    0.12
        gas               600.00    0.97
        hydro           33830.00  122.36
        win

In [15]:
# --- PRINT CAPACITIES H2 ---
print("--- H2 OPTIMAL CAPACITIES (MW/MWh) ---")

h2_links = nd.links[nd.links.carrier == "H2"]
if not h2_links.empty:
    print("\nElectrolyzer & Fuel Cell capacities (MW):")
    print(h2_links[['p_nom_opt']])

h2_stores = nd.stores[nd.stores.carrier == "H2"]
if not h2_stores.empty:
    print("\nStorage capacities (MWh):")
    print(h2_stores[['e_nom_opt']])

h2_pipes = nd.links[nd.links.carrier == "H2 pipeline"]
if not h2_pipes.empty:
    print("\nPipeline capacities (MW):")
    print(h2_pipes[['p_nom_opt']])

--- H2 OPTIMAL CAPACITIES (MW/MWh) ---

Electrolyzer & Fuel Cell capacities (MW):
                  p_nom_opt
name                       
DK electrolyzer    0.000000
DK fuel cell       0.000000
DE electrolyzer    0.000000
DE fuel cell       0.000000
NO electrolyzer   20.108947
NO fuel cell     717.099760
SE electrolyzer    0.000000
SE fuel cell       0.000000

Storage capacities (MWh):
              e_nom_opt
name                   
DK H2 store  878.254481
DE H2 store  876.242127
NO H2 store  876.800027
SE H2 store  879.931325

Pipeline capacities (MW):
                               p_nom_opt
name                                    
H2 pipeline Denmark-Germany   876.242127
H2 pipeline Germany-Denmark   875.365885
H2 pipeline Denmark-Norway   1200.952453
H2 pipeline Norway-Denmark    887.586416
H2 pipeline Denmark-Sweden    585.362009
H2 pipeline Sweden-Denmark    719.731057
H2 pipeline Germany-Sweden      0.000000
H2 pipeline Sweden-Germany      0.000000
H2 pipeline Germany-Norway    

In [16]:


# =========================
# 1. IDENTIFY COMPONENTS
# =========================
ch4_market_gens = nd.generators.index[nd.generators.carrier == "gas market"]
methanation_links = nd.links.index[nd.links.carrier == "CH4"]
gas_links = nd.links.index[nd.links.carrier == "gas"]
ch4_pipeline_links = nd.links.index[nd.links.carrier == "CH4 pipeline"]

print("Gas market generators:")
print(ch4_market_gens.tolist(), "\n")

print("Methanation links:")
print(methanation_links.tolist(), "\n")

print("Gas plant links:")
print(gas_links.tolist(), "\n")

print("CH4 pipeline links:")
print(ch4_pipeline_links.tolist(), "\n")


# =========================
# 2. ANNUAL ENERGY BY SOURCE / USE
# =========================
# Gas market: generator output on CH4 buses
market_twh = (nd.generators_t.p[ch4_market_gens].sum() / 1e6).rename("TWh")

# Methanation: electricity converted into CH4
# p1 is usually the output side of the link (bus1 = CH4 bus)
methanation_twh = (-nd.links_t.p1[methanation_links].sum() / 1e6).rename("TWh")

# Gas plants: CH4 consumed to generate electricity
# p0 is input side (bus0 = CH4 bus), so use abs or minus sign depending on convention
gas_use_twh = (nd.links_t.p0[gas_links].abs().sum() / 1e6).rename("TWh")

# Pipelines: transported CH4 through each link
pipeline_twh = (nd.links_t.p0[ch4_pipeline_links].abs().sum() / 1e6).rename("TWh")


# =========================
# 3. COUNTRY-LEVEL TABLES
# =========================
market_df = pd.DataFrame({
    "country": nd.generators.loc[ch4_market_gens, "bus"].str.replace(" CH4", "", regex=False),
    "TWh_from_market": market_twh
}).set_index("country").sort_index()

metha_df = pd.DataFrame({
    "country": nd.links.loc[methanation_links, "bus1"].str.replace(" CH4", "", regex=False),
    "TWh_from_methanation": methanation_twh
}).set_index("country").sort_index()

gas_df = pd.DataFrame({
    "country": nd.links.loc[gas_links, "bus0"].str.replace(" CH4", "", regex=False),
    "TWh_to_gas_plants": gas_use_twh
}).set_index("country").sort_index()

pipe_df = pd.DataFrame({
    "link": ch4_pipeline_links,
    "TWh_pipeline_flow": pipeline_twh
}).sort_values("TWh_pipeline_flow", ascending=False)

country_summary = (
    market_df.join(metha_df, how="outer")
             .join(gas_df, how="outer")
             .fillna(0)
             .round(3)
)

print("=== CH4 COUNTRY SUMMARY (TWh/year) ===")
print(country_summary)
print()

print("=== CH4 PIPELINE FLOWS BY LINK (TWh/year, absolute) ===")
print(pipe_df.round(3))
print()


# =========================
# 4. TOTALS
# =========================
total_market = market_df["TWh_from_market"].sum()
total_metha = metha_df["TWh_from_methanation"].sum()
total_gas_use = gas_df["TWh_to_gas_plants"].sum()
total_pipeline = pipe_df["TWh_pipeline_flow"].sum()

totals = pd.Series({
    "CH4 from market (TWh)": round(total_market, 3),
    "CH4 from methanation (TWh)": round(total_metha, 3),
    "CH4 consumed in gas plants (TWh)": round(total_gas_use, 3),
    "CH4 transported in pipelines (gross, TWh)": round(total_pipeline, 3),
})

print("=== CH4 TOTALS ===")
print(totals)
print()


# =========================
# 5. OPTIONAL: SHARE OF MARKET VS METHANATION
# =========================
total_supply = total_market + total_metha

if total_supply > 0:
    shares = pd.Series({
        "Market share of CH4 supply (%)": round(100 * total_market / total_supply, 2),
        "Methanation share of CH4 supply (%)": round(100 * total_metha / total_supply, 2),
    })
    print("=== CH4 SUPPLY SHARES ===")
    print(shares)

Gas market generators:
['DK CH4 market', 'DE CH4 market', 'NO CH4 market', 'SE CH4 market'] 

Methanation links:
['DK methanation', 'DE methanation', 'NO methanation', 'SE methanation'] 

Gas plant links:
['DE_gas', 'NO_gas', 'SE_gas', 'DK_gas'] 

CH4 pipeline links:
['CH4 pipeline Denmark-Germany', 'CH4 pipeline Germany-Denmark', 'CH4 pipeline Denmark-Norway', 'CH4 pipeline Norway-Denmark', 'CH4 pipeline Denmark-Sweden', 'CH4 pipeline Sweden-Denmark', 'CH4 pipeline Germany-Sweden', 'CH4 pipeline Sweden-Germany', 'CH4 pipeline Germany-Norway', 'CH4 pipeline Norway-Germany'] 

=== CH4 COUNTRY SUMMARY (TWh/year) ===
         TWh_from_market  TWh_from_methanation  TWh_to_gas_plants
country                                                          
Denmark            0.000                   0.0              3.185
Germany            9.009                  -0.0              2.577
Norway             5.256                  -0.0              0.971
Sweden             0.000                  -0.0  